##### Modelo KMV — Probabilidad de Incumplimiento Histórica | 5 Emisores Colombianos | 2017–2022
EEFF desde XBRL/SIMEV (Arelle) · Precios desde Yahoo Finance · Solver Merton → DD/PD = Φ(−DD)


In [1]:
import warnings; warnings.filterwarnings('ignore')
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator
from scipy.stats import norm
from scipy.optimize import fsolve
import requests, urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from arelle import Cntlr

# ── Rutas base (portables — funcionan al mover el proyecto) ─
try:
    # VS Code Jupyter: apunta siempre al directorio del .ipynb
    HERE = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    # Jupyter Lab / classic: usa el CWD del servidor
    HERE = Path().resolve()

RUTA_SALIDA    = HERE                  # salidas junto al notebook
RUTA_EEFF_BASE = HERE.parent / 'EEFF' # carpeta hermana con los XBRL
RUTA_PASIVOS_CSV = RUTA_SALIDA / 'eeff_pasivos_kmv.csv'

print(f"✓ Directorio base  : {HERE}")
print(f"  EEFF (XBRL)      : {RUTA_EEFF_BASE}")
print(f"  Salida archivos  : {RUTA_SALIDA}")

# ── Emisores: nombre → carpeta SIMEV ────────────────────────
EMISORES_XBRL = {
    'Ecopetrol':       'ECOPETROL',
    'Bancolombia':     'BANCOLOMBIA',
    'Banco de Bogota': 'BANCOBOGOTA',
    'Cementos Argos':  'CEMARGOS',
    'Grupo Nutresa':   'GRUPO NUTRESA',
}

# Colores para gráficos
COLORES = {
    'Ecopetrol':       '#f7931a',
    'Bancolombia':     '#00d4ff',
    'Banco de Bogota': '#7c3aed',
    'Cementos Argos':  '#10b981',
    'Grupo Nutresa':   '#f43f5e',
}

FECHA_INICIO = '2015-01-01'   # para precios (ventana de volatilidad)
FECHA_FIN    = '2021-12-31'
TASA_RF      = 0.05            # tasa libre de riesgo Colombia aprox.
T_HORIZONTE  = 1               # horizonte 1 año (KMV estándar)

# ── Sesión HTTP con SSL bypass (Yahoo Finance) ────────────────
SESSION = requests.Session()
SESSION.verify = False
SESSION.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json',
})

# ── Estética dark theme ──────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#0d1117',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': 'white',
    'xtick.color': 'white',        'ytick.color':     'white',
    'text.color':  'white',        'grid.color':      '#21262d',
    'legend.facecolor': '#161b22', 'legend.edgecolor':'#30363d',
})


✓ Directorio base  : C:\Users\MELISSA\OneDrive - Universidad EIA\Documents\Proceso Gómez\mad_estructurador\parte_1
  EEFF (XBRL)      : C:\Users\MELISSA\OneDrive - Universidad EIA\Documents\Proceso Gómez\mad_estructurador\EEFF
  Salida archivos  : C:\Users\MELISSA\OneDrive - Universidad EIA\Documents\Proceso Gómez\mad_estructurador\parte_1


##### Sección 1 — Extracción de EEFF vía XBRL (SIMEV/RNVE)
Tags IFRS con esquema primario → secundario (fallback). Default Point: `DP = PC + 0,5 × PNC`.


In [2]:

# ============================================================
# CELDA 2 — DEFINICIÓN DE TAGS IFRS Y FUNCIONES AUXILIARES
# ============================================================
# Cada tag tiene un 'primary' (preferido) y 'secondary' (fallbacks).
# Para cuentas de balance (is_balance=True) filtramos por endDate
# y priorizamos hechos SIN dimensiones (renglón total, no desagregaciones
# del Estado de Cambios en Patrimonio ni columnas de segmentos).

TAGS_KMV = {
    # ── Balance General ─────────────────────────────────────
    'pasivo_total': {
        'is_balance': True,
        'primary':    'ifrs:Liabilities',
        'secondary':  [],
    },
    'pasivo_corriente': {
        'is_balance': True,
        'primary':    'ifrs:CurrentLiabilities',
        'secondary':  [],
    },
    'pasivo_no_corriente': {
        'is_balance': True,
        'primary':    'ifrs:NoncurrentLiabilities',
        'secondary':  [],
    },
    'patrimonio': {
        'is_balance': True,
        'primary':    'ifrs:Equity',
        'secondary':  [],
    },
    'activo_total': {
        'is_balance': True,
        'primary':    'ifrs:Assets',
        'secondary':  [],
    },
    # ── Estado de Resultados ─────────────────────────────────
    'ingreso_neto': {
        'is_balance': False,
        'primary':    'ifrs:ProfitLoss',
        'secondary':  ['ifrs:ComprehensiveIncome'],
    },
    'eps': {
        'is_balance': False,
        'primary':    'ifrs:DilutedEarningsLossPerShare',
        'secondary':  ['ifrs:BasicEarningsLossPerShare',
                       'ifrs:BasicAndDilutedEarningsLossPerShare'],
    },
}

# Conjunto de todos los tags para filtrado rápido al iterar facts
TAGS_INTERES = set()
for cfg in TAGS_KMV.values():
    TAGS_INTERES.add(cfg['primary'])
    TAGS_INTERES.update(cfg['secondary'])


def buscar_valor(factData, tag_name, fecha_obj, is_balance):
    """
    Busca el valor numérico de un tag IFRS en el DataFrame de facts.

    Para cuentas de balance (is_balance=True):
      - Filtra por endDate == fecha_obj  (o fecha_obj + 1 día, por convención XBRL)
      - Prioriza hechos sin dimensiones (renglón total del balance)

    Para flujos (is_balance=False):
      - Devuelve el primer valor no nulo disponible
    """
    if factData.empty:
        return None

    mask = (factData['name'] == tag_name) & (factData['isNumeric'])

    if is_balance:
        fecha_p1 = fecha_obj + pd.Timedelta(days=1)
        mask_fecha = (factData['endDate'] == fecha_obj) | (factData['endDate'] == fecha_p1)
        mask = mask & mask_fecha

        # Preferir sin dimensiones (total del balance, no componentes del ECPN)
        sin_dim = factData[mask & factData['sin_dimensiones']]['value'].dropna()
        sin_dim = sin_dim[sin_dim != 0]
        if len(sin_dim) > 0:
            return float(sin_dim.iloc[0])

    candidatos = factData[mask]['value'].dropna()
    candidatos = candidatos[candidatos != 0]
    return float(candidatos.iloc[0]) if len(candidatos) > 0 else None


print(f"✓ Tags IFRS definidos: {len(TAGS_KMV)} campos | lookup set: {len(TAGS_INTERES)} tags")


✓ Tags IFRS definidos: 7 campos | lookup set: 10 tags


In [3]:
BASE_XBRL = RUTA_EEFF_BASE
CORTES_VALIDOS = {'03-31', '06-30', '09-30', '12-31'}
FECHA_MIN = pd.Timestamp('2017-01-01')
FECHA_MAX = pd.Timestamp('2021-12-31')


def xbrl_a_dataframe(modelo):
    registros = []
    for fact in modelo.factsInInstance:
        ctx = fact.context
        if ctx is None:
            continue
        try:
            if ctx.isInstantPeriod:
                end_date = pd.Timestamp(str(ctx.instantDatetime.date()))
            elif ctx.isStartEndPeriod:
                end_date = pd.Timestamp(str(ctx.endDatetime.date()))
            else:
                continue
        except Exception:
            continue

        qname = str(fact.concept.qname)
        if qname not in TAGS_INTERES:
            continue

        try:
            valor = float(fact.xValue) if fact.isNumeric else None
        except (TypeError, ValueError):
            valor = None

        registros.append({
            'name': qname,
            'value': valor,
            'endDate': end_date,
            'isNumeric': fact.isNumeric,
            'sin_dimensiones': len(ctx.qnameDims) == 0,
        })

    return pd.DataFrame(registros) if registros else pd.DataFrame(
        columns=['name', 'value', 'endDate', 'isNumeric', 'sin_dimensiones']
    )


def extraer_emisor(carpeta_emisor, nombre_emisor, cntlr):
    archivos = sorted(carpeta_emisor.glob('*.xbrl'))
    filas = []

    for arch in archivos:
        fecha_str = arch.stem.split('_')[-1]
        try:
            fecha = pd.Timestamp(fecha_str)
        except Exception:
            continue

        if fecha_str[5:] not in CORTES_VALIDOS:
            continue
        if not (FECHA_MIN <= fecha <= FECHA_MAX):
            continue

        try:
            model = cntlr.modelManager.load(str(arch))
        except Exception:
            continue

        factData = xbrl_a_dataframe(model)

        fila = {'emisor': nombre_emisor, 'fecha': fecha_str}
        for campo, cfg in TAGS_KMV.items():
            val = buscar_valor(factData, cfg['primary'], fecha, cfg['is_balance'])
            if val is None:
                for sec in cfg['secondary']:
                    val = buscar_valor(factData, sec, fecha, cfg['is_balance'])
                    if val is not None:
                        break
            fila[campo] = val

        filas.append(fila)
        model.close()
        gc.collect()

    return filas


cntlr = Cntlr.Cntlr(logFileName='logToPrint')
cntlr.startLogging(logFileName='logToPrint')

todas_filas = []
for emisor_key, carpeta_nombre in EMISORES_XBRL.items():
    carpeta = BASE_XBRL / carpeta_nombre
    filas_emisor = extraer_emisor(carpeta, emisor_key, cntlr)
    todas_filas.extend(filas_emisor)

cntlr.close()
gc.collect()

df_eeff = pd.DataFrame(todas_filas)
df_eeff.sort_values(['emisor', 'fecha'], inplace=True)
df_eeff.reset_index(drop=True, inplace=True)

OUT_EEFF = RUTA_SALIDA / "eeff_kmv_completo.csv"
df_eeff.to_csv(OUT_EEFF, index=False)

print(f"EEFF extraído: {len(df_eeff)} filas")


2026-04-18 10:00:57,992 [xmlSchema:valueError] Element ifrs:DividendsPaidOrdinaryShares type decimal value error: 135321496, lexical pattern mismatch - 0052031821_0043_000005_0000_000000_000000_C-I_2019-06-30.xbrl 40288

2026-04-18 10:00:57,992 [xmlSchema:valueError] Element ifrs:DividendsPaidOrdinaryShares type decimal value error: 135321496, lexical pattern mismatch - 0052031821_0043_000005_0000_000000_000000_C-I_2019-06-30.xbrl 40288

2026-04-18 10:00:57,998 [xmlSchema:valueError] Element ifrs:DividendsPaidOtherShares type decimal value error: 24580747, lexical pattern mismatch - 0052031821_0043_000005_0000_000000_000000_C-I_2019-06-30.xbrl 40289

2026-04-18 10:00:57,998 [xmlSchema:valueError] Element ifrs:DividendsPaidOtherShares type decimal value error: 24580747, lexical pattern mismatch - 0052031821_0043_000005_0000_000000_000000_C-I_2019-06-30.xbrl 40289

2026-04-18 10:00:57,998 [xmlSchema:valueError] Element ifrs:DividendsPaidOrdinarySharesPerShare type decimal value error: 60

##### Sección 2 — Precios de Mercado y Volatilidad del Equity
Yahoo Finance 2016–2022 · ADR EC (×20) y CIB (×4) convertidos a COP con TRM BanRep · σ_E = std(log-ret) × √252, ventana 252 días.


In [ ]:
import yfinance as yf

TICKERS_YF = {
    'ECOPETROL': 'EC',
    'BANCOLOMBIA': 'CIB',
    'BANCOBOGOTA': 'BOGOTA.CL',
    'CEMARGOS': 'CEMARGOS.CL',
    'GRUPONUTRESA': 'NUTRESA.CL',
}

ADR_RATIO = {'ECOPETROL': 20, 'BANCOLOMBIA': 4}
FECHA_INICIO_YF = '2016-01-01'
FECHA_FIN_YF = '2022-06-30'


def descargar_precios(ticker):
    """
    Descarga precios usando yf.Ticker().history() — más robusto que yf.download()
    en versiones recientes de yfinance.
    """
    try:
        tk = yf.Ticker(ticker)
        df = tk.history(start=FECHA_INICIO_YF, end=FECHA_FIN_YF, auto_adjust=True)
        if df is None or df.empty:
            print(f"  ✗ Sin datos para {ticker}")
            return pd.DataFrame()
        df = df[['Close']].rename(columns={'Close': 'close'})
        # Eliminar timezone si existe (tz-naive para compatibilidad con merge)
        if df.index.tz is not None:
            df.index = df.index.tz_localize(None)
        df.index = pd.to_datetime(df.index)
        df.index.name = 'fecha'
        return df.dropna()
    except Exception as e:
        print(f"  ✗ Error descargando {ticker}: {e}")
        return pd.DataFrame()


# ── TRM (USD/COP) ─────────────────────────────────────────────────────────────
df_trm_raw = descargar_precios('COP=X')
if df_trm_raw.empty:
    print("⚠ TRM no disponible — se usará TRM fija aproximada (4200 COP/USD)")
    idx = pd.date_range(FECHA_INICIO_YF, FECHA_FIN_YF, freq='B')
    df_trm = pd.DataFrame({'trm': 4200.0}, index=idx)
    df_trm.index.name = 'fecha'
else:
    df_trm = df_trm_raw.rename(columns={'close': 'trm'})

# ── Descargar precios por emisor ──────────────────────────────────────────────
precios_dict = {}
for emisor, ticker in TICKERS_YF.items():
    df_px = descargar_precios(ticker)
    if df_px.empty:
        print(f"  ⚠ Saltando {emisor} ({ticker}) — sin datos")
        continue

    if emisor in ADR_RATIO:
        df_px = df_px.join(df_trm, how='left')
        df_px['trm'] = df_px['trm'].ffill().bfill()
        df_px['close'] = df_px['close'] * df_px['trm'] / ADR_RATIO[emisor]
        df_px = df_px[['close']]

    df_px['emisor'] = emisor
    precios_dict[emisor] = df_px
    print(f"  ✓ {emisor:20s} ({ticker}): {len(df_px)} filas")

if not precios_dict:
    raise RuntimeError(
        "No se pudo descargar ningún precio. "
        "Verifica la conexión a internet y la versión de yfinance."
    )

df_precios = pd.concat(precios_dict.values())
df_precios.to_csv(RUTA_SALIDA / "precios_kmv.csv")

print(f"\n✓ Precios descargados: {len(df_precios)} filas | {df_precios['emisor'].nunique()} emisores")



1 Failed download:
['COP=X']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['EC']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['CIB']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BOGOTA.CL']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['CEMARGOS.CL']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['NUTRESA.CL']: TypeError("'NoneType' object is not subscriptable")


ValueError: No objects to concatenate

##### Sección 3 — Modelo Merton–KMV: Solver y Resultados
Equity modelado como opción de compra sobre activos (Merton 1974). `fsolve` → (V_A, σ_A) → DD = ln(V_A/DP) / (σ_A√T) → PD = Φ(−DD).


In [ ]:
VENTANA_VOL = 252

dfs_vol = []
for emisor, df_px in precios_dict.items():
    df_v = df_px[['close']].copy()
    df_v['log_ret'] = np.log(df_v['close'] / df_v['close'].shift(1))
    df_v['sigma_E_diaria'] = df_v['log_ret'].rolling(VENTANA_VOL, min_periods=120).std()
    df_v['sigma_E'] = df_v['sigma_E_diaria'] * np.sqrt(252)
    df_v['emisor_key'] = emisor
    dfs_vol.append(df_v)

df_vol = pd.concat(dfs_vol).reset_index()
df_vol['fecha'] = pd.to_datetime(df_vol['fecha'])

EMISOR_TO_KEY = {
    'ecopetrol': 'ECOPETROL',
    'bancolombia': 'BANCOLOMBIA',
    'banco de bogota': 'BANCOBOGOTA',
    'cementos argos': 'CEMARGOS',
    'grupo nutresa': 'GRUPONUTRESA',
}


def normalizar_emisor(nombre):
    return str(nombre).strip().lower()


def precio_en_fecha(df_vol_emisor, fecha_ts, tol_dias=5):
    sub = df_vol_emisor[df_vol_emisor['fecha'] <= fecha_ts].tail(tol_dias)
    if sub.empty:
        return np.nan, np.nan
    row = sub.iloc[-1]
    if (fecha_ts - row['fecha']).days > tol_dias:
        return np.nan, np.nan
    return row['close'], row['sigma_E']


registros_kmv = []
for _, row in df_eeff.iterrows():
    emisor = row['emisor']
    emisor_key = EMISOR_TO_KEY.get(normalizar_emisor(emisor), emisor)
    fecha_ts = pd.Timestamp(row['fecha'])

    df_em = df_vol[df_vol['emisor_key'] == emisor_key]
    close, sigma_e = precio_en_fecha(df_em, fecha_ts)

    if pd.notna(row['eps']) and row['eps'] != 0 and pd.notna(row['ingreso_neto']) and row['ingreso_neto'] != 0:
        num_acciones = abs(row['ingreso_neto'] / row['eps'])
    else:
        num_acciones = np.nan

    E_market = num_acciones * close if (pd.notna(num_acciones) and pd.notna(close)) else np.nan

    pc = row.get('pasivo_corriente', np.nan)
    pnc = row.get('pasivo_no_corriente', np.nan)
    pt = row.get('pasivo_total', np.nan)

    if pd.notna(pc) and pd.notna(pnc):
        DP = pc + 0.5 * pnc
        dp_metodo = 'std+0.5*ltd'
        dp_proxy = False
    elif pd.notna(pt):
        DP = pt * 0.75
        dp_metodo = 'proxy_0.75_total'
        dp_proxy = True
    else:
        DP = np.nan
        dp_metodo = 'sin_dato'
        dp_proxy = True

    registros_kmv.append({
        'emisor': emisor,
        'emisor_key': emisor_key,
        'fecha': row['fecha'],
        'E_market': E_market,
        'sigma_E': sigma_e,
        'DP': DP,
        'dp_metodo': dp_metodo,
        'dp_proxy': dp_proxy,
        'num_acciones': num_acciones,
        'close': close,
        'pasivo_corriente': pc,
        'pasivo_no_corriente': pnc,
        'pasivo_total': pt,
        'patrimonio': row.get('patrimonio'),
        'activo_total': row.get('activo_total'),
        'ingreso_neto': row.get('ingreso_neto'),
        'eps': row.get('eps'),
    })

df_kmv = pd.DataFrame(registros_kmv)
df_kmv['fecha'] = pd.to_datetime(df_kmv['fecha'])
df_kmv.sort_values(['emisor', 'fecha'], inplace=True)
df_kmv.reset_index(drop=True, inplace=True)

print(f"KMV base: {len(df_kmv)} filas")


In [ ]:
def merton_system(pars, E, sigma_E, DP, r, T):
    V_A, sigma_A = pars
    if V_A <= 0 or sigma_A <= 0:
        return [1e10, 1e10]
    d1 = (np.log(V_A / DP) + (r + 0.5 * sigma_A**2) * T) / (sigma_A * np.sqrt(T))
    d2 = d1 - sigma_A * np.sqrt(T)
    eq1 = V_A * norm.cdf(d1) - DP * np.exp(-r * T) * norm.cdf(d2) - E
    eq2 = norm.cdf(d1) * sigma_A * V_A - sigma_E * E
    return [eq1, eq2]


def resolver_merton(E, sigma_E, DP, r=TASA_RF, T=T_HORIZONTE):
    if any(pd.isna(x) or x <= 0 for x in [E, sigma_E, DP]):
        return np.nan, np.nan

    V0 = E + DP
    s0 = sigma_E * E / V0
    try:
        sol = fsolve(
            merton_system,
            [V0, s0],
            args=(E, sigma_E, DP, r, T),
            full_output=True,
        )
        V_A, sigma_A = sol[0]
        if V_A > 0 and sigma_A > 0:
            return V_A, sigma_A
    except Exception:
        pass
    return np.nan, np.nan


resultados = df_kmv.apply(
    lambda row: pd.Series(
        resolver_merton(row['E_market'], row['sigma_E'], row['DP']),
        index=['V_A', 'sigma_A'],
    ),
    axis=1,
)
df_kmv[['V_A', 'sigma_A']] = resultados

print(f"✓ Solver Merton: convergencia {df_kmv['V_A'].notna().sum()}/{len(df_kmv)}")


In [ ]:
def calcular_dd_pd(row, r=TASA_RF, T=T_HORIZONTE):
    V_A = row['V_A']
    sigma_A = row['sigma_A']
    DP = row['DP']
    if any(pd.isna(x) or x <= 0 for x in [V_A, sigma_A, DP]):
        return pd.Series({'DD': np.nan, 'PD': np.nan})

    DD = (np.log(V_A / DP) + (r - 0.5 * sigma_A**2) * T) / (sigma_A * np.sqrt(T))
    PD = norm.cdf(-DD)
    return pd.Series({'DD': DD, 'PD': PD})


df_kmv[['DD', 'PD']] = df_kmv.apply(calcular_dd_pd, axis=1)

OUT_KMV = RUTA_SALIDA / "kmv_resultados.csv"
df_kmv.to_csv(OUT_KMV, index=False)

print(f"✓ DD/PD calculados | guardado: {OUT_KMV.name}")


##### Sección 4 — Figura 1: Evolución histórica de la PD (5 emisores)
PD trimestral Merton–KMV para Bancobogotá, Bancolombia, Cemargos, Ecopetrol y Grupo Nutresa (2017 Q1 – 2022 Q1). Eje Y acotado en 13 %.


In [ ]:
# Solo periodos válidos de PD
plot_df = df_kmv.dropna(subset=['PD']).copy()
plot_df = plot_df[(plot_df['PD'] >= 0) & (plot_df['PD'] <= 1)]

# Estilo similar al ejemplo solicitado
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(13, 6.5), dpi=140)

COLORES_FIG = {
    'ECOPETROL':    '#2ca02c',
    'BANCOLOMBIA':  '#ff7f0e',
    'BANCOBOGOTA':  '#7f7f7f',
    'CEMARGOS':     '#f1c40f',
    'GRUPONUTRESA': '#000000',
}

NOMBRES_DISPLAY = {
    'ECOPETROL':    'Ecopetrol',
    'BANCOLOMBIA':  'Bancolombia',
    'BANCOBOGOTA':  'Banco de Bogotá',
    'CEMARGOS':     'Cementos Argos',
    'GRUPONUTRESA': 'Grupo Nutresa',
}

for emisor_key, sub in plot_df.groupby('emisor_key'):
    sub = sub.sort_values('fecha')
    if sub.empty:
        continue

    nombre = NOMBRES_DISPLAY.get(emisor_key, emisor_key)
    color = COLORES_FIG.get(emisor_key, None)

    ax.plot(
        sub['fecha'],
        sub['PD'] * 100,
        marker='o',
        markersize=4,
        linewidth=1.8,
        label=nombre,
        color=color,
        alpha=0.9
    )

ax.set_title('Figura 1: Evolución probabilidad de incumplimiento múltiples emisores.', fontsize=12, weight='bold')
ax.set_xlabel('Fecha EEFF', fontsize=12)
ax.set_ylabel('Prob incumplimiento', fontsize=12)

y_max = 13.0
ax.set_ylim(0, y_max)
ax.yaxis.set_major_locator(MultipleLocator(1.0))
ax.yaxis.set_minor_locator(MultipleLocator(0.5))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}%'))

# Eje X
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=8)

ax.legend(loc='upper left', ncol=5, fontsize=8, frameon=False)
ax.grid(True, which='major', linestyle='--', alpha=0.4)
ax.grid(True, which='minor', linestyle=':', alpha=0.25)

plt.tight_layout()

OUT_FIG = RUTA_SALIDA / "figura1_kmv_pd.png"
fig.savefig(OUT_FIG, bbox_inches='tight')
plt.show()
plt.show()

##### Sección 5 — Literal b: PD vs Calificación Crediticia (Ecopetrol)
PD KMV (eje izq.) vs calificación local de largo plazo Fitch Colombia (eje der., escala ordinal). Evalúa si el modelo anticipa revisiones de rating.


In [ ]:
EMISOR_ANALISIS = 'ECOPETROL'
NOMBRE_ANALISIS = 'Ecopetrol'

pd_emisor = (
    df_kmv[df_kmv['emisor_key'] == EMISOR_ANALISIS][['fecha', 'PD']]
    .dropna()
    .sort_values('fecha')
    .copy()
)
pd_emisor['PD_pct'] = pd_emisor['PD'] * 100

RUTA_RATING = RUTA_SALIDA / "rating_ecopetrol_historico.csv"

if RUTA_RATING.exists():
    df_rating = pd.read_csv(RUTA_RATING)
    df_rating['fecha'] = pd.to_datetime(df_rating['fecha'], errors='coerce', dayfirst=True)
    df_rating = df_rating[df_rating['fecha'].notna()]
    # Solo conservar filas con rating_label completado
    df_rating = df_rating[df_rating['rating_label'].notna() & (df_rating['rating_label'].astype(str).str.strip() != '')]
    df_rating = df_rating.sort_values('fecha').reset_index(drop=True)
else:
    df_rating = pd.DataFrame({
        'fecha': pd.date_range('2017-12-31', periods=6, freq='YE'),
        'agencia': 'BRC S&P',
        'rating_label': None,
        'perspectiva': None,
    })
    df_rating.to_csv(RUTA_RATING, index=False)

print(f"✓ PD Ecopetrol: {len(pd_emisor)} periodos | ratings disponibles: {len(df_rating)}")
df_rating[['fecha', 'agencia', 'rating_label', 'perspectiva']].head(10)


In [ ]:
RATING_SCORE = {
    'AAA': 1, 'AA+': 2, 'AA': 3, 'AA-': 4,
    'A+': 5, 'A': 6, 'A-': 7,
    'BBB+': 8, 'BBB': 9, 'BBB-': 10,
    'BB+': 11, 'BB': 12, 'BB-': 13,
    'B+': 14, 'B': 15, 'B-': 16,
    'CCC+': 17, 'CCC': 18, 'CCC-': 19,
    'CC': 20, 'C': 21, 'D': 22,
}

RATING_INV = {v: k for k, v in RATING_SCORE.items()}

df_rating_plot = df_rating.copy()
df_rating_plot['rating_label'] = df_rating_plot['rating_label'].astype(str).str.upper().str.strip()
df_rating_plot['rating_score'] = df_rating_plot['rating_label'].map(RATING_SCORE)
df_rating_plot = df_rating_plot.dropna(subset=['rating_score']).sort_values('fecha')

fig, ax1 = plt.subplots(figsize=(12, 6), dpi=140)

ax1.plot(pd_emisor['fecha'], pd_emisor['PD_pct'], color='#ff7f0e', marker='o', linewidth=2, label='PD KMV (%)')
ax1.set_ylabel('PD KMV (%)', color='#ff7f0e')
ax1.tick_params(axis='y', labelcolor='#ff7f0e')
ax1.set_xlabel('Fecha')
ax1.set_title('Ecopetrol: Probabilidad de incumplimiento (KMV) vs calificación de crédito (Fitch local)')
ax1.grid(True, linestyle='--', alpha=0.35)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

ax2 = ax1.twinx()
if not df_rating_plot.empty:
    ax2.step(
        df_rating_plot['fecha'],
        df_rating_plot['rating_score'],
        where='post',
        color='#1f77b4',
        linewidth=2,
        label='Rating Fitch (escala ordinal)'
    )
    ticks = sorted(df_rating_plot['rating_score'].unique())
    ax2.set_yticks(ticks)
    ax2.set_yticklabels([RATING_INV[int(t)] for t in ticks])
else:
    print("⚠ No hay ratings disponibles — completar rating_ecopetrol_historico.csv")

ax2.invert_yaxis()
ax2.set_ylabel('Calificación (mejor arriba)')

lineas = ax1.get_lines() + ax2.get_lines()
labels = [l.get_label() for l in lineas]
ax1.legend(lineas, labels, loc='upper left')

plt.tight_layout()
OUT_FIG2 = RUTA_SALIDA / "figura2_ecopetrol_pd_rating.png"
fig.savefig(OUT_FIG2, bbox_inches='tight')
print(f"✓ Figura 2 guardada: {OUT_FIG2.name}")
plt.show()


In [ ]:
# ============================================================
# EXTRACCIÓN COMPLEMENTARIA — Tags adicionales solo Ecopetrol
# Para el Punto 2: Revenue, EBIT, Caja, Activo Corriente, Costos Financieros
# No modifica el pipeline principal de 5 emisores.
# ============================================================

TAGS_ECO_EXTRA = {
    'activo_corriente': {
        'is_balance': True,
        'primary':   'ifrs:CurrentAssets',
        'secondary': [],
    },
    'caja': {
        'is_balance': True,
        'primary':   'ifrs:CashAndCashEquivalents',
        'secondary': [],
    },
    'ingresos': {
        'is_balance': False,
        'primary':   'ifrs:Revenue',
        'secondary': ['ifrs:RevenueFromContractsWithCustomers'],
    },
    'ebit': {
        'is_balance': False,
        'primary':   'ifrs:ProfitLossFromOperatingActivities',
        'secondary': ['ifrs:OperatingIncomeLoss'],
    },
    'costos_financieros': {
        'is_balance': False,
        'primary':   'ifrs:FinanceCosts',
        'secondary': ['ifrs:InterestExpense'],
    },
    'depreciacion': {
        'is_balance': False,
        'primary':   'ifrs:DepreciationAndAmortisationExpense',
        'secondary': ['ifrs:DepreciationAmortisationAndImpairmentLossReversalOfImpairmentLossRecognisedInProfitOrLoss'],
    },
}

TAGS_INTERES_EXTRA = set()
for _cfg in TAGS_ECO_EXTRA.values():
    TAGS_INTERES_EXTRA.add(_cfg['primary'])
    TAGS_INTERES_EXTRA.update(_cfg['secondary'])

def xbrl_a_dataframe_extra(modelo):
    """Igual que xbrl_a_dataframe pero filtra por TAGS_INTERES_EXTRA."""
    registros = []
    for fact in modelo.factsInInstance:
        ctx = fact.context
        if ctx is None:
            continue
        try:
            if ctx.isInstantPeriod:
                end_date = pd.Timestamp(str(ctx.instantDatetime.date()))
            elif ctx.isStartEndPeriod:
                end_date = pd.Timestamp(str(ctx.endDatetime.date()))
            else:
                continue
        except Exception:
            continue

        qname = str(fact.concept.qname)
        if qname not in TAGS_INTERES_EXTRA:
            continue

        try:
            valor = float(fact.xValue) if fact.isNumeric else None
        except (TypeError, ValueError):
            valor = None

        registros.append({
            'name': qname,
            'value': valor,
            'endDate': end_date,
            'isNumeric': fact.isNumeric,
            'sin_dimensiones': len(ctx.qnameDims) == 0,
        })

    return pd.DataFrame(registros) if registros else pd.DataFrame(
        columns=['name', 'value', 'endDate', 'isNumeric', 'sin_dimensiones']
    )

# Parsear solo carpeta ECOPETROL
carpeta_eco = BASE_XBRL / "ECOPETROL"
archivos_eco = sorted(carpeta_eco.glob('*.xbrl'))

cntlr2 = Cntlr.Cntlr(logFileName='logToPrint')
cntlr2.startLogging(logFileName='logToPrint')

filas_extra = []
for arch in archivos_eco:
    fecha_str = arch.stem.split('_')[-1]
    try:
        fecha = pd.Timestamp(fecha_str)
    except Exception:
        continue
    if fecha_str[5:] not in CORTES_VALIDOS:
        continue
    if not (FECHA_MIN <= fecha <= FECHA_MAX):
        continue

    try:
        model = cntlr2.modelManager.load(str(arch))
    except Exception:
        continue

    factData = xbrl_a_dataframe_extra(model)
    fila = {'fecha': fecha_str}
    for campo, cfg in TAGS_ECO_EXTRA.items():
        val = buscar_valor(factData, cfg['primary'], fecha, cfg['is_balance'])
        if val is None:
            for sec in cfg['secondary']:
                val = buscar_valor(factData, sec, fecha, cfg['is_balance'])
                if val is not None:
                    break
        fila[campo] = val

    filas_extra.append(fila)
    model.close()
    gc.collect()

cntlr2.close()
gc.collect()

df_eco_extra = pd.DataFrame(filas_extra).sort_values('fecha').reset_index(drop=True)

# EBITDA estimado = EBIT + Depreciación (cuando ambos disponibles)
df_eco_extra['ebitda'] = df_eco_extra['ebit'] + df_eco_extra['depreciacion'].fillna(0)
# Si depreciacion no está disponible, ebitda = ebit como proxy
mask_sin_dep = df_eco_extra['depreciacion'].isna()
df_eco_extra.loc[mask_sin_dep, 'ebitda'] = df_eco_extra.loc[mask_sin_dep, 'ebit']

print(f"✓ Tags extra Ecopetrol: {len(df_eco_extra)} periodos")
print(f"  Disponibilidad por campo:")
for col in ['activo_corriente','caja','ingresos','ebit','costos_financieros','depreciacion','ebitda']:
    n = df_eco_extra[col].notna().sum()
    print(f"    {col:25s}: {n}/{len(df_eco_extra)} periodos")
df_eco_extra.head()


##### Sección 6 — Punto 2: Análisis Histórico de Métricas EEFF vs PD (Ecopetrol)
Cuatro paneles: Deuda Neta/EBITDA · Cobertura de intereses (EBIT/Int.) · Márgenes (EBITDA y Neto) · Liquidez (Razón Corriente y Caja/Activos).


In [ ]:
EMISOR_P2_KEY   = 'ECOPETROL'
EMISOR_P2_LABEL = 'ECOPETROL'
NOMBRE_P2       = 'Ecopetrol'

# ── Base EEFF existente + tags extra ──────────────────────────────────────────
base = (
    df_eeff[df_eeff['emisor'].str.upper().str.contains(EMISOR_P2_LABEL)]
    .copy()
    .sort_values('fecha')
)
base['fecha'] = pd.to_datetime(base['fecha'])

eco_extra = df_eco_extra.copy()
eco_extra['fecha'] = pd.to_datetime(eco_extra['fecha'])

metricas = pd.merge(base, eco_extra, on='fecha', how='left')

if metricas.empty:
    raise ValueError(f'No hay datos de EEFF para {NOMBRE_P2}.')

# ── Ratios de endeudamiento ───────────────────────────────────────────────────
metricas['deuda_ebitda']     = metricas['pasivo_total']  / metricas['ebitda']
metricas['ebit_intereses']   = metricas['ebit']          / metricas['costos_financieros'].abs()
metricas['pasivo_patrimonio']= metricas['pasivo_total']  / metricas['patrimonio']
metricas['pasivo_activo']    = metricas['pasivo_total']  / metricas['activo_total']

# ── Ratios de rentabilidad ────────────────────────────────────────────────────
metricas['margen_ebitda']    = metricas['ebitda']        / metricas['ingresos']
metricas['margen_neto']      = metricas['ingreso_neto']  / metricas['ingresos']
metricas['roa']              = metricas['ingreso_neto']  / metricas['activo_total']

# ── Ratios de liquidez ────────────────────────────────────────────────────────
metricas['razon_corriente']  = metricas['activo_corriente'] / metricas['pasivo_corriente']
metricas['caja_activos']     = metricas['caja']          / metricas['activo_total']

# ── PD del modelo KMV ────────────────────────────────────────────────────────
pd_p2 = (
    df_kmv[df_kmv['emisor_key'] == EMISOR_P2_KEY][['fecha', 'PD']]
    .copy().sort_values('fecha')
)
pd_p2['fecha'] = pd.to_datetime(pd_p2['fecha'])

analisis_p2 = pd.merge(metricas, pd_p2, on='fecha', how='inner')
analisis_p2['PD_pct']          = analisis_p2['PD']           * 100
analisis_p2['margen_ebitda_pct']= analisis_p2['margen_ebitda']* 100
analisis_p2['margen_neto_pct'] = analisis_p2['margen_neto']  * 100
analisis_p2['caja_activos_pct']= analisis_p2['caja_activos'] * 100

# ── Correlaciones con PD ──────────────────────────────────────────────────────
corr_cols = ['PD','deuda_ebitda','ebit_intereses','pasivo_patrimonio',
             'margen_ebitda','margen_neto','roa','razon_corriente','caja_activos']
corr_pd = (analisis_p2[corr_cols]
           .corr(numeric_only=True)['PD']
           .drop('PD')
           .sort_values(ascending=False))

# ── Exportar ──────────────────────────────────────────────────────────────────
OUT_P2_TABLA = RUTA_SALIDA / 'punto2_ecopetrol_analisis.csv'
OUT_P2_CORR  = RUTA_SALIDA / 'punto2_ecopetrol_correlaciones_pd.csv'
analisis_p2.to_csv(OUT_P2_TABLA, index=False)
corr_pd.rename('corr_con_pd').to_csv(OUT_P2_CORR, index=True)

# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 3 — Panel 4 en 1: KMV-style inspirado en la figura del enunciado
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(4, 1, figsize=(13, 16), dpi=140, sharex=True)
fecha = analisis_p2['fecha']

# Panel 1: PD vs Deuda/EBITDA
ax0 = axes[0]
ax0b = ax0.twinx()
ax0.bar(fecha, analisis_p2['ebitda'] / 1e9, width=60, color='#2c3e50', alpha=0.75, label='EBITDA (MM COP miles MM)')
ax0b.plot(fecha, analisis_p2['deuda_ebitda'], color='#f39c12', marker='o', linewidth=2, label='Deuda Bruta / EBITDA')
ax0b.plot(fecha, analisis_p2['PD_pct'],       color='#e74c3c', marker='^', linewidth=2, linestyle='--', label='PD KMV (%)')
ax0.set_ylabel('EBITDA (miles MM COP)')
ax0b.set_ylabel('Deuda/EBITDA  |  PD (%)')
ax0.set_title('Figura 3 — Ecopetrol: Métricas EEFF vs PD (KMV)', fontsize=12, weight='bold')
lines0 = ax0.patches[:1] + ax0b.get_lines()
labels0 = ['EBITDA'] + [l.get_label() for l in ax0b.get_lines()]
ax0.legend(lines0, labels0, loc='upper left', fontsize=7, frameon=False)
ax0.grid(True, linestyle='--', alpha=0.3)

# Panel 2: Cobertura de intereses (EBIT/Intereses)
ax1 = axes[1]
ax1b = ax1.twinx()
ax1.plot(fecha, analisis_p2['ebit_intereses'], color='#1f77b4', marker='o', linewidth=2, label='EBIT / Intereses (cobertura)')
ax1b.plot(fecha, analisis_p2['PD_pct'], color='#e74c3c', marker='^', linewidth=2, linestyle='--', label='PD KMV (%)')
ax1.axhline(1.5, color='gray', linestyle=':', linewidth=1, alpha=0.6)
ax1.set_ylabel('Cobertura de Intereses (x)')
ax1b.set_ylabel('PD (%)')
ax1.legend(loc='upper left', fontsize=7, frameon=False)
ax1b.legend(loc='upper right', fontsize=7, frameon=False)
ax1.grid(True, linestyle='--', alpha=0.3)
ax1.set_title('Cobertura de Intereses (EBIT / Gastos Financieros)')

# Panel 3: Márgenes de rentabilidad
ax2 = axes[2]
ax2b = ax2.twinx()
ax2.plot(fecha, analisis_p2['margen_ebitda_pct'], color='#2ca02c', marker='o', linewidth=2, label='Margen EBITDA (%)')
ax2.plot(fecha, analisis_p2['margen_neto_pct'],   color='#9467bd', marker='s', linewidth=2, label='Margen Neto (%)')
ax2b.plot(fecha, analisis_p2['PD_pct'], color='#e74c3c', marker='^', linewidth=2, linestyle='--', label='PD KMV (%)')
ax2.set_ylabel('Margen (%)')
ax2b.set_ylabel('PD (%)')
ax2.legend(loc='upper left', fontsize=7, frameon=False)
ax2b.legend(loc='upper right', fontsize=7, frameon=False)
ax2.grid(True, linestyle='--', alpha=0.3)
ax2.set_title('Rentabilidad: Margen EBITDA y Margen Neto')

# Panel 4: Liquidez
ax3 = axes[3]
ax3b = ax3.twinx()
ax3.plot(fecha, analisis_p2['razon_corriente'],  color='#17becf', marker='o', linewidth=2, label='Razón Corriente (x)')
ax3.plot(fecha, analisis_p2['caja_activos_pct'], color='#bcbd22', marker='s', linewidth=2, label='Caja / Activos (%)')
ax3b.plot(fecha, analisis_p2['PD_pct'], color='#e74c3c', marker='^', linewidth=2, linestyle='--', label='PD KMV (%)')
ax3.set_ylabel('Liquidez (x  |  %)')
ax3b.set_ylabel('PD (%)')
ax3.legend(loc='upper left', fontsize=7, frameon=False)
ax3b.legend(loc='upper right', fontsize=7, frameon=False)
ax3.grid(True, linestyle='--', alpha=0.3)
ax3.set_title('Liquidez: Razón Corriente y Caja / Activos Totales')

ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
for lbl in ax3.get_xticklabels():
    lbl.set_rotation(45); lbl.set_ha('right')

plt.tight_layout()
OUT_FIG3 = RUTA_SALIDA / 'figura3_ecopetrol_metricas_pd.png'
fig.savefig(OUT_FIG3, bbox_inches='tight')
print(f'✓ Figura 3 guardada: {OUT_FIG3.name}')
print(f'✓ Correlaciones con PD:')
display(corr_pd.to_frame('corr_con_pd'))


In [ ]:
# Comparación explícita: periodos de mayor y menor PD vs métricas clave (Ecopetrol)
peak   = analisis_p2.loc[analisis_p2['PD'].idxmax()]
min_pd = analisis_p2.loc[analisis_p2['PD'].idxmin()]

display(pd.DataFrame({
    'Periodo':             [peak['fecha'].date(),                  min_pd['fecha'].date()],
    'PD (%)':              [f"{peak['PD_pct']:.2f}",               f"{min_pd['PD_pct']:.4f}"],
    'Deuda / EBITDA':      [f"{peak['deuda_ebitda']:.2f}x",        f"{min_pd['deuda_ebitda']:.2f}x"],
    'Cobert. Intereses':   [f"{peak['ebit_intereses']:.2f}x",      f"{min_pd['ebit_intereses']:.2f}x"],
    'Margen EBITDA (%)':   [f"{peak['margen_ebitda_pct']:.1f}",    f"{min_pd['margen_ebitda_pct']:.1f}"],
    'Razón Corriente':     [f"{peak['razon_corriente']:.2f}x",     f"{min_pd['razon_corriente']:.2f}x"],
}, index=['Mayor PD', 'Menor PD']))
